# PD-L1 Stomach — 면역세포 통합 5-class 계층형 학습

림프구/대식세포를 **Immune**으로 합칩니다. 상피세포/면역세포/기타 root와
상피세포 및 면역세포의 PD-L1 음성/양성 분기만 학습합니다.
면역세포 subtype head와 loss는 없습니다.

| ID | Class | 원본 라벨 |
|---:|---|---|
| 0 | Negative Epithelial | Negative Epithelial |
| 1 | Negative Immune | Negative Lymphocyte + Negative Macrophage |
| 2 | Positive Epithelial | Positive Epithelial |
| 3 | Positive Immune | Positive Lymphocyte + Positive Macrophage |
| 4 | other | other |

위에서부터 순서대로 실행하세요. GPU_ID는 기본 1입니다.
처음부터 학습하며 기존 실험과 별도 폴더에 저장합니다.
검증은 최종 5개 클래스의 Macro F1과 Detection Recall을 사용합니다.


In [ ]:
import os
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm
import yaml

from nets import nn
from utils import util
from utils.ihc_dataset import (
    IHCHierarchicalDataset,
    PDL1_CLASS_NAMES,
    count_record_classes,
    discover_records,
    slide_id_from_record,
    split_records_by_slide_balanced,
)
from utils.valid import (
    compute_point_label_metrics_single,
    plot_training_progress,
    visualize_ground_truth_and_prediction_separately_detail_single,
)

warnings.filterwarnings('ignore')
GPU_ID = 0
if torch.cuda.is_available():
    if GPU_ID >= torch.cuda.device_count():
        raise RuntimeError(
            f'GPU_ID={GPU_ID}, available GPUs={torch.cuda.device_count()}'
        )
    torch.cuda.set_device(GPU_ID)
    device = torch.device(f'cuda:{GPU_ID}')
    print(f'device: {device} | {torch.cuda.get_device_name(GPU_ID)}')
else:
    device = torch.device('cpu')
    print('device: cpu')

with open('utils/detail_args.yaml', errors='ignore') as file:
    params = yaml.safe_load(file)

from utils.pdl1_merged_immune import (
    CLASS_NAMES, CLASS_COLORS, SOURCE_TO_MERGED, MergedImmuneDataset,
    yolo_v11_m_pdl1_merged_immune, PDL1MergedImmuneLoss,
)
params['names'] = CLASS_NAMES


In [ ]:
def collate_fn(batch):
    return IHCHierarchicalDataset.collate_fn(batch)


input_size = 512
resize_size = 1024
data_root = '../../data/mediautoAI_PDL1_Stomach/'
VALIDATION_SLIDE_COUNT = 3

records = discover_records(data_root)
train_records, val_records = split_records_by_slide_balanced(
    records, val_slide_count=VALIDATION_SLIDE_COUNT
)
train_slides = {slide_id_from_record(record) for record in train_records}
val_slides = {slide_id_from_record(record) for record in val_records}
assert train_slides.isdisjoint(val_slides)

# 80%의 crop은 원본의 희소 클래스(1~5)를 중앙부에 포함하도록 선택합니다.
# 나머지 20%는 완전 random crop이므로 자연 분포도 유지합니다.
CLASS_AWARE_CROP_PROBABILITY = 0.8
CROP_CLASS_WEIGHTS = [1, 1, 1, 1, 1, 1, 1]

train_base = IHCHierarchicalDataset(
    train_records, input_size=input_size, resize_size=resize_size,
    augment=True, random_crop=True, cache_images=True,
    class_aware_crop_probability=CLASS_AWARE_CROP_PROBABILITY,
    class_sampling_weights=CROP_CLASS_WEIGHTS,
)
val_base = IHCHierarchicalDataset(
    val_records, input_size=input_size, resize_size=resize_size,
    augment=False, random_crop=False, cache_images=True, four_crop=True,
)
train_dataset = MergedImmuneDataset(train_base)
val_dataset = MergedImmuneDataset(val_base)

source_counts = count_record_classes(train_records).astype(np.int64)
merged_counts = np.zeros(5, dtype=np.int64)
for source_id, merged_id in enumerate(SOURCE_TO_MERGED):
    merged_counts[merged_id] += source_counts[source_id]
for class_id, class_name in CLASS_NAMES.items():
    print(f'  [{class_id}] {class_name:24s}: {merged_counts[class_id]:,}')

root_counts = np.array([
    merged_counts[0] + merged_counts[2],
    merged_counts[1] + merged_counts[3], merged_counts[4],
])
hierarchy_params = {
    **params, 'objectness': 1.0, 'root': 2.0,
    'epithelial': 1.0, 'immune_polarity': 2.0,
    'root_class_weights': np.sqrt(root_counts.max() / np.maximum(root_counts, 1)).tolist(),
    'epithelial_positive_weight': float(np.sqrt(merged_counts[0] / max(merged_counts[2], 1))),
    'immune_negative_weight': float(np.sqrt(merged_counts[3] / max(merged_counts[1], 1))),
    'objectness_focal_gamma': 1.5,
}

EPOCH_DATA_REPEATS = 50
batch_size = 16
train_epoch_dataset = torch.utils.data.ConcatDataset(
    [train_dataset] * EPOCH_DATA_REPEATS
)
train_loader = torch.utils.data.DataLoader(
    train_epoch_dataset, batch_size=batch_size, shuffle=True, num_workers=4,
    collate_fn=collate_fn, drop_last=False,
    pin_memory=(device.type == 'cuda'), persistent_workers=True,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=4,
    collate_fn=collate_fn, drop_last=False,
    pin_memory=(device.type == 'cuda'), persistent_workers=True,
)

print(f'\nTrain source patches : {len(train_dataset):,} ({len(train_slides)} slides)')
print(f'Samples/epoch        : {len(train_epoch_dataset):,}')
print(f'Train batches        : {len(train_loader):,}')
print(f'Val source/tiles     : {len(val_records)} / {len(val_dataset)}')
cache_mib = (train_dataset.cache_memory_bytes + val_dataset.cache_memory_bytes) / 1024**2
print(f'1024 image cache     : {cache_mib:.1f} MiB')


In [ ]:
# 매 호출마다 random crop이 실제로 바뀌는지 간단히 확인합니다.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis in axes:
    image, classes, boxes, _ = train_dataset[0]
    axis.imshow(image.permute(1, 2, 0).numpy())
    for class_id, box in zip(classes.long(), boxes):
        color = CLASS_COLORS[int(class_id)]
        axis.scatter(
            box[0].item() * input_size, box[1].item() * input_size,
            facecolors='none', edgecolors=color, s=18, linewidths=1,
        )
    axis.set_title(' | '.join(f'{CLASS_NAMES[c]}: {(classes == c).sum().item()}' for c in CLASS_NAMES), fontsize=7)
    axis.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

LEARNING_RATE = 5e-4
MIN_LEARNING_RATE = 1e-6
MAX_EPOCHS = 2000

model = yolo_v11_m_pdl1_merged_immune().to(device)
optimizer = torch.optim.AdamW(
    util.set_params(model, params['weight_decay']),
    lr=LEARNING_RATE, betas=(0.9, 0.999),
    weight_decay=params['weight_decay'],
)
criterion = PDL1MergedImmuneLoss(model, hierarchy_params)
scheduler = CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE
)

model.train()
smoke_images, smoke_targets = next(iter(train_loader))
smoke_images = smoke_images.to(device).float() / 255.0
with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
    smoke_outputs = model(smoke_images)
    smoke_losses = criterion(smoke_outputs, smoke_targets)
print('Loss smoke test: ' + ' | '.join(f'{k}={v.item():.4f}' for k, v in smoke_losses.items()))
del smoke_images, smoke_targets, smoke_outputs, smoke_losses
if device.type == 'cuda':
    torch.cuda.empty_cache()

print('Model: hierarchical YOLOv11-m, 5 classes, no immune subtype head')
print(f'Epochs={MAX_EPOCHS}, data repeats/epoch={EPOCH_DATA_REPEATS}')
print('Early stopping: disabled')


In [ ]:
train_losses = []
val_det_recalls = []
val_cls_accs = []
val_macro_precisions = []
val_macro_recalls = []
val_macro_f1s = []
val_class_stats_history = []

save_dir = '../../model/mediautoAI_PDL1_Stomach/pdl1_merged_immune_hierarchical/'
os.makedirs(save_dir, exist_ok=True)
start_epoch = 0
best_validation_score = 0.0
accumulate = max(round(64 / batch_size), 1)
amp_scale = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
VALIDATION_CONFIDENCE_THRESHOLD = 0.10

# 재개할 때 아래 블록의 주석을 해제합니다.
# checkpoint_path = os.path.join(save_dir, 'last_model.pt')
# if os.path.exists(checkpoint_path):
#     checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
#     if checkpoint.get('architecture') != 'pdl1_merged_immune_hierarchical':
#         raise ValueError('Merged Immune checkpoint가 아닙니다.')
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
#     amp_scale.load_state_dict(checkpoint['amp_scale_state_dict'])
#     start_epoch = checkpoint['epoch'] + 1
#     best_validation_score = checkpoint.get('best_validation_score', 0.0)

print(f'Gradient accumulation steps: {accumulate}')

for epoch in range(start_epoch, MAX_EPOCHS):
    model.train()
    loss_meters = {name: util.AverageMeter() for name in ('box', 'objectness', 'root', 'epithelial', 'immune_polarity', 'dfl')}
    avg_total = util.AverageMeter()
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(enumerate(train_loader), total=len(train_loader))
    for step, (images, targets) in progress:
        images = images.to(device, non_blocking=True).float() / 255.0
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            losses = criterion(outputs, targets)
            unscaled_loss = losses['total']
            loss = unscaled_loss / accumulate

        batch_n = images.size(0)
        for name, meter in loss_meters.items():
            meter.update(losses[name].item(), batch_n)
        avg_total.update(unscaled_loss.item(), batch_n)
        amp_scale.scale(loss).backward()
        if (step + 1) % accumulate == 0 or (step + 1) == len(train_loader):
            amp_scale.step(optimizer)
            amp_scale.update()
            optimizer.zero_grad(set_to_none=True)

        memory = (
            f'{torch.cuda.memory_reserved() / 1e9:.3f}G'
            if device.type == 'cuda' else 'CPU'
        )
        progress.set_description(
            f'Epoch {epoch + 1}/{MAX_EPOCHS} | Memory {memory} | '
            + ' | '.join(f'{name} {meter.avg:.3f}' for name, meter in loss_meters.items())
        )

    scheduler.step()
    train_losses.append(avg_total.avg)

    try:
        metrics = compute_point_label_metrics_single(
            model, val_loader, device, params, distance_threshold=16,
            class_agnostic_nms=True,
            confidence_threshold=VALIDATION_CONFIDENCE_THRESHOLD,
        )
        detection_recall = float(metrics.get('detection_recall', 0))
        cls_accuracy = float(metrics.get('classification_accuracy', 0))
        macro_precision = float(metrics.get('macro_precision', 0))
        macro_recall = float(metrics.get('macro_recall', 0))
        macro_f1 = float(metrics.get('macro_f1', 0))
        class_stats = metrics.get('class_stats', {})
        validation_score = 0.7 * macro_f1 + 0.3 * detection_recall
    except Exception as error:
        print(f'Validation 오류: {error}')
        detection_recall = cls_accuracy = 0.0
        macro_precision = macro_recall = macro_f1 = 0.0
        validation_score = 0.0
        class_stats = {}

    val_det_recalls.append(detection_recall)
    val_cls_accs.append(cls_accuracy)
    val_macro_precisions.append(macro_precision)
    val_macro_recalls.append(macro_recall)
    val_macro_f1s.append(macro_f1)
    val_class_stats_history.append(class_stats)

    print(f'\nEpoch {epoch + 1}/{MAX_EPOCHS}')
    print(f'  Detection Recall : {detection_recall:.4f}')
    print(f'  Class Accuracy   : {cls_accuracy:.4f}')
    print(f'  Macro P/R/F1     : {macro_precision:.4f} / {macro_recall:.4f} / {macro_f1:.4f}')
    for name, stats in class_stats.items():
        print(
            f"  {name:24s} P={stats['precision']:.4f} | "
            f"R={stats['recall']:.4f} | F1={stats['f1']:.4f}"
        )

    improved = validation_score > best_validation_score
    if improved:
        best_validation_score = validation_score

    checkpoint = {
        'architecture': 'pdl1_merged_immune_hierarchical',
        'class_mapping': CLASS_NAMES,
        'source_to_merged': SOURCE_TO_MERGED,
        'num_classes': 5,
        'epoch': epoch,
        'epoch_data_repeats': EPOCH_DATA_REPEATS,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'amp_scale_state_dict': amp_scale.state_dict(),
        'train_loss': avg_total.avg,
        **{f'{name}_loss': meter.avg for name, meter in loss_meters.items()},
        'detection_recall': detection_recall,
        'classification_accuracy': cls_accuracy,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'validation_score': validation_score,
        'best_validation_score': best_validation_score,
        'class_stats': class_stats,
        'params': hierarchy_params,
    }
    torch.save(checkpoint, os.path.join(save_dir, 'last_model.pt'))
    if improved:
        torch.save(checkpoint, os.path.join(save_dir, 'best_model.pt'))
        print(f'  New best validation score: {validation_score:.4f}')

    if (epoch + 1) % 10 == 0:
        try:
            sample_index = random.randrange(len(val_dataset))
            visualize_ground_truth_and_prediction_separately_detail_single(
                model, val_dataset, idx=sample_index, epoch=epoch + 1,
                save_dir=save_dir, class_names=CLASS_NAMES,
                class_colors=CLASS_COLORS,
                conf_threshold=VALIDATION_CONFIDENCE_THRESHOLD,
            )
            plot_training_progress(
                train_losses, val_det_recalls, val_cls_accs,
                val_macro_precisions, val_macro_recalls, val_macro_f1s,
                epoch + 1, save_dir,
                class_stats_history=val_class_stats_history,
            )
        except Exception as error:
            print(f'시각화 오류: {error}')

print('Merged Immune 학습 완료')
print(f'Best validation score: {best_validation_score:.4f}')
print(f'Save directory: {save_dir}')
